In [ ]:
# Imports
import sys
import os
sys.path.append(os.path.abspath('../..'))


from controller.marl.main import setup_language_analysis
from controller.marl.config import MappoConfig
from controller.marl.run_sim import run_sim

from project_paths import PROJECT_ROOT


import torch
from controller.marl.datasets import ObsData
from torch.utils.data import DataLoader


from plt_style import set_style
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
set_style()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
config = MappoConfig.from_yaml(PROJECT_ROOT / "configs" / "mappo_settings.yaml")

In [ ]:
system, config = setup_language_analysis(config, device)

In [ ]:
run_sim(system, config, device, 10, collect_obs_file="./temp.csv")

In [ ]:
obs_logs_file = "./temp.csv"

dataset = ObsData(obs_logs_file, system["act_shape"][0])

dataloader = DataLoader(dataset, batch_size=config.aim_batch_size, shuffle=True)

In [ ]:
aim = system["aim"].to(device)

In [ ]:
node_losses = torch.zeros(system["agent_obs_shape"][0], device=device)

total_samples = 0
for batch_obs, batch_rewards in dataloader:

    batch_obs = batch_obs.to(device)
    current_batch_size = batch_obs.size(0)

    node_losses += aim.get_individual_losses(batch_obs, None) * current_batch_size
    total_samples += current_batch_size

node_losses /= total_samples

In [ ]:
# Bar Chart
data = node_losses.detach().cpu().numpy()
feature_indices = [f"Feature {i}" for i in range(len(data))]

df = pd.DataFrame({
    'Feature': feature_indices,
    'Loss': data
})

plt.figure(figsize=(10, 12))
ax = sns.barplot(x='Loss', y='Feature', data=df, palette='viridis')

ax.set_title('Individual Node Losses breakdown')
ax.set_xlabel('Mean Reconstruction Loss')
ax.set_ylabel('Observation Feature Index')

plt.show()

In [ ]:
for index, i in enumerate(aim.obs_mask):
    if not i:
        print(index, i)

In [ ]:


# OBS_DIM = system['agent_obs_shape'][0]
# COMM_DIM = config.communication_size
# VOCAB_SIZE = config.vocab_size
# NC = config.num_comms
# ACTION_COUNT = system['act_shape'][0]
# HQ_LAYERS = config.hq_layers


# # aim = AIM(
# #     OBS_DIM, ACTION_COUNT,
# #     512, COMM_DIM * NC, VOCAB_SIZE,
# #     commitment_cost=0.25,
# #     num_training_steps=num_training_steps, hq_vae=HQ_LAYERS,
# #     init_temperature=config.init_temperature, min_temperature=config.min_temperature,
# #     autoencoder_type=config.autoencoder_type, obs_mask=torch.tensor(obs_mask, dtype=torch.bool, device=device)
# # ).to(device)

# optimizer = torch.optim.Adam(aim.parameters(), lr=config.aim_learning_rate)

# for epoch in range(config.ae_epochs):

#     train_loss = 0.0
#     aim.train()

#     for batch_obs, batch_rewards in training_loader:

#         batch_obs = batch_obs.to(device)
#         batch_rewards = batch_rewards.to(device)

#         optimizer.zero_grad()
#         loss = aim(batch_obs, None)
#         loss.backward()

#         torch.nn.utils.clip_grad_norm_(aim.parameters(), max_norm=1.0)

#         optimizer.step()
#         train_loss += loss.item()
        


# # save language
# aim.save_encoder()